In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import time
import numpy as np
import os
import pandas as pd
pd.options.display.max_columns = 100
pd.options.display.max_rows = 130

In [9]:
from utils_data import (
    load_raw_data, check_dups)
from settings_lt_shorts import (
    audio_settings, create_directories, load_data_settings, lt_shorts_categories, load_video_configs
)
from utils_lt_shorts import (
    stitch_audios, draw_lt_vocab_list_whole_image, create_video_with_concat_images
)

# Get data

In [11]:
truly_load_data = True
if truly_load_data:
    df_all_vocab = load_raw_data()
    df_all_vocab.to_csv('static/latest_data.csv', index=False)
else:
    df_all_vocab = pd.read_csv('static/latest_data.csv')
    print('!!!!!!!! WARNING: not truly loading data !!!!!!!!')

df_dups = check_dups(df_all_vocab)
print(df_all_vocab.shape)
print(f'# duplicate vocab: {len(df_dups)}')
df_all_vocab.head(3)

(7600, 38)
# duplicate vocab: 0


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,1,房贷,fáng dài,mortgage,word,1,life,NaN,Finance & Economy,NaN,借贷与负债,MISSING,1.0,1.0,2.0,1.0,房子,house,贷款,loan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他每个月都要还房贷,Ta měi gè yuè dōu yào huán fángdài,He has to pay his mortgage every month,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,2,白天,bái tiān,daytime,word,2,time,NaN,Time,NaN,时间段 / 时长,1,2.0,1.0,1.0,1.0,白,white,天,day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,白天很热晚上比较凉快,Báitiān hěn rè wǎnshàng bǐjiào liángkuai,It is hot in the daytime and cooler at night,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,3,组成,zǔ chéng,to form;make up,word,3,general,NaN,Language & Expression,NaN,逻辑与结构,MISSING,5.0,5.0,5.0,3.0,组,set,成,become,NaN,NaN,NaN,NaN,NaN,NaN,NaN,水是由氢和氧组成的,Shuǐ shì yóu qīng hé yǎng zǔchéng de,Water is made up of hydrogen and oxygen,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


# Get settings

In [5]:
# '', '', ''
current_category = '宠物'
current_category_dict = lt_shorts_categories[current_category]
data_settings = load_data_settings(current_category_dict)
create_directories(data_settings)
data_settings

{'chinese': '宠物',
 'pinyin': 'chǒng wù',
 'english': 'pets',
 'vocab_list': [['兽医', '绝育', '项圈'], ['毛皮', '翻肚皮', '抚摸'], ['流浪狗', '铲屎官', '舔毛']],
 'replacements': {'collar (pet)': 'collar',
  'to roll over (show belly)': 'roll over',
  'shit': 'poop'},
 'output_path': 'output/lt_shorts/宠物',
 'output_path_audio': 'output/lt_shorts/宠物/audio_files',
 'output_path_images': 'output/lt_shorts/宠物/images'}

In [6]:
# get list of all need for audio
df_for_audio = df_all_vocab[df_all_vocab['chinese'].isin([x for y in data_settings['vocab_list'] for x in y])].reset_index(drop=True)
all_audios = [data_settings['chinese']] + df_for_audio['chinese'].tolist() + df_for_audio['sentence'].tolist() + df_for_audio['word1'].tolist() + df_for_audio['word2'].tolist() + df_for_audio['word3'].tolist() + df_for_audio['word4'].tolist()
all_audios = [x for x in list(set(all_audios)) if not pd.isna(x)]
print(all_audios)

all_audios_english = df_for_audio['english'].tolist()
all_audios_english

['项圈', '猫翻肚皮要你摸。', '皮', '他轻轻抚摸小猫', '兽', '肚皮', '官', '铲', '摸', '舔毛', '流浪', '铲屎官', '抚摸', '圈', '绝育', '我作为铲屎官每天都很忙。', '医', '舔', '翻过来', '宠物', '猫生病了要带去看兽医。', '毛皮外套很保暖', '猫经常自己舔毛。', '兽医', '项', '屎', '毛皮', '流浪狗', '我给猫买了一个新项圈。', '翻肚皮', '狗', '拒绝', '猫绝育后会更安静。', '生育', '街上有一只流浪狗', '毛', '抚慰']


['stray dog',
 'fur',
 'to pet;caress',
 'collar (pet)',
 'veterinarian',
 'to sterilize;spay',
 'cat owner;pooper scooper',
 'to groom',
 'to roll over (show belly)']

In [7]:
if data_settings['vocab_list'][0].__class__ == list:
    n_parts = len(data_settings['vocab_list'])
    data_settings['n_parts'] = n_parts
else:
    n_parts = 1

if n_parts > 1:
    for i in range(n_parts):
        df_vocab_list = df_all_vocab[df_all_vocab['chinese'].isin(data_settings['vocab_list'][i])].reset_index(drop=True)
        # Sort in order
        df_vocab_list['word_order'] = df_vocab_list['chinese'].map({x: i_x for i_x, x in enumerate(data_settings['vocab_list'][i])})
        df_vocab_list = df_vocab_list.sort_values('word_order').reset_index(drop=True)
        if 'replacements' in data_settings.keys():
            df_vocab_list = df_vocab_list.replace(data_settings['replacements'])
        data_settings['current_part'] = i + 1
        print(f'Processing part {i+1} of {n_parts}')
        df_durations = stitch_audios(audio_settings, data_settings, None, df_vocab_list, part_number=i+1)
        video_configs = load_video_configs()
        final_img_file_path = draw_lt_vocab_list_whole_image(video_configs, data_settings, df_vocab_list)
        create_video_with_concat_images(df_durations, df_vocab_list, data_settings)
else:
    df_vocab_list = df_all_vocab[df_all_vocab['chinese'].isin(data_settings['vocab_list'])].reset_index(drop=True)
    df_durations = stitch_audios(audio_settings, data_settings, None, df_vocab_list)
    video_configs = load_video_configs()
    final_img_file_path = draw_lt_vocab_list_whole_image(video_configs, data_settings, df_vocab_list)
    create_video_with_concat_images(df_durations, df_vocab_list, data_settings)


Processing part 1 of 3


/var/folders/z8/s7_mn6894xd5kj7pdpx6t6100000gn/T/ipykernel_79851/2302433815.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vocab_list = df_vocab_list.replace(data_settings['replacements'])


Audio duration: 31.0s
Adding clip: output/lt_shorts/宠物/images/title_only.png for duration 1.9s
Adding clip: output/lt_shorts/宠物/images/vocab_word_0_component_only.png for duration 2.7s
Adding clip: output/lt_shorts/宠物/images/vocab_word_0_full.png for duration 3.7s
Adding clip: output/lt_shorts/宠物/images/vocab_word_0_sentence.png for duration 2.9s
Adding clip: output/lt_shorts/宠物/images/vocab_word_1_component_only.png for duration 3.7s
Adding clip: output/lt_shorts/宠物/images/vocab_word_1_full.png for duration 4.4s
Adding clip: output/lt_shorts/宠物/images/vocab_word_1_sentence.png for duration 2.5s
Adding clip: output/lt_shorts/宠物/images/vocab_word_2_component_only.png for duration 3.3s
Adding clip: output/lt_shorts/宠物/images/vocab_word_2_full.png for duration 3.5s
Adding clip: output/lt_shorts/宠物/images/vocab_word_2_sentence.png for duration 2.4s
Final audio duration: 31.100s
Final video duration before audio set: 31.036s
MoviePy - Building video output/lt_shorts/宠物/宠物_video_part1.mp4.
M

MoviePy - Done.
MoviePy - Writing video output/lt_shorts/宠物/宠物_video_part1.mp4



/var/folders/z8/s7_mn6894xd5kj7pdpx6t6100000gn/T/ipykernel_79851/2302433815.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vocab_list = df_vocab_list.replace(data_settings['replacements'])


...

MoviePy - Done.
MoviePy - Writing video output/lt_shorts/宠物/宠物_video_part2.mp4



/var/folders/z8/s7_mn6894xd5kj7pdpx6t6100000gn/T/ipykernel_79851/2302433815.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vocab_list = df_vocab_list.replace(data_settings['replacements'])


MoviePy - Done !
MoviePy - video ready output/lt_shorts/宠物/宠物_video_part2.mp4
Processing part 3 of 3
Audio duration: 32.1s
Adding clip: output/lt_shorts/宠物/images/title_only.png for duration 1.9s
Adding clip: output/lt_shorts/宠物/images/vocab_word_0_component_only.png for duration 2.8s
Adding clip: output/lt_shorts/宠物/images/vocab_word_0_full.png for duration 3.9s
Adding clip: output/lt_shorts/宠物/images/vocab_word_0_sentence.png for duration 2.1s
Adding clip: output/lt_shorts/宠物/images/vocab_word_1_component_only.png for duration 4.6s
Adding clip: output/lt_shorts/宠物/images/vocab_word_1_full.png for duration 4.7s
Adding clip: output/lt_shorts/宠物/images/vocab_word_1_sentence.png for duration 3.0s
Adding clip: output/lt_shorts/宠物/images/vocab_word_2_component_only.png for duration 3.5s
Adding clip: output/lt_shorts/宠物/images/vocab_word_2_full.png for duration 3.4s
Adding clip: output/lt_shorts/宠物/images/vocab_word_2_sentence.png for duration 2.1s
Final audio duration: 32.160s
Final video 

MoviePy - Done.
MoviePy - Writing video output/lt_shorts/宠物/宠物_video_part3.mp4



MoviePy - Done !
MoviePy - video ready output/lt_shorts/宠物/宠物_video_part3.mp4


In [8]:
# from moviepy import ImageClip
# img_show = ImageClip(final_img_file_path, duration=1).with_start(0)
# img_show.display_in_notebook()